# Normalizacion en SQL

### Escenarios e importancia

Escenarios donde se suele utilizar la normalización:

* **Integridad de datos:** Imagina una base de datos que contiene información de clientes. Sin normalización, si un cliente cambia su edad, tendríamos que actualizarla en varios lugares, lo que aumentaría el riesgo de inconsistencias. Al normalizar los datos, podemos tener tablas separadas vinculadas por un identificador único que garantizará que los datos se mantengan precisos y coherentes.
* **Eficiencia en las consultas:** Consideremos una base de datos compleja con múltiples tablas relacionadas que almacena información redundante. En este escenario, las consultas que involucran uniones (*joins*) se vuelven más complicadas y consumen más recursos. La normalización ayudará a simplificar las consultas al dividir los datos en tablas más pequeñas, donde cada tabla contiene solo información relevante, reduciendo así la necesidad de uniones complejas.
* **Optimización del almacenamiento:** Un problema importante con los datos redundantes es que ocupan espacio de almacenamiento innecesario. Por ejemplo, si almacenamos los mismos detalles del producto en cada registro de pedido, se produce una duplicación. Con la normalización, puedes eliminar la redundancia dividiendo los datos en tablas separadas.

¿Por qué es importante la normalización en SQL?:

* **Reduce la redundancia:** La redundancia ocurre cuando la misma información se almacena varias veces; una buena forma de evitarlo es dividiendo los datos en tablas más pequeñas.
* **Mejora el rendimiento de las consultas:** Puedes ejecutar consultas de forma más rápida en tablas pequeñas que han sido sometidas a un proceso de normalización.
* **Minimiza las anomalías de actualización:** Con tablas normalizadas, puedes actualizar los datos fácilmente sin afectar a otros registros.
* **Mejora la integridad de los datos:** Garantiza que la información se mantenga coherente y precisa.

### Niveles de Normalización

**1NF (Primera Forma Normal)**

Este nivel de normalización garantiza que cada columna de tus datos contenga únicamente valores atómicos. En este contexto, valores atómicos significa que cada entrada en una columna es indivisible. Es como decir que cada celda de una hoja de cálculo debe contener solo una pieza de información. La 1NF asegura la atomicidad de los datos, con cada celda de la columna conteniendo un solo valor y cada columna con un nombre único.

**2NF (Segunda Forma Normal)**

La Segunda Forma Normal elimina las dependencias parciales al garantizar que los atributos que no son clave dependan únicamente de la clave primaria. Lo que esto significa, en esencia, es que debe haber una relación directa entre cada columna y la clave primaria, y no entre otras columnas.

**3NF (Tercera Forma Normal)**

La 3NF se basa en la 2NF al eliminar las dependencias transitivas. Una dependencia transitiva ocurre cuando un atributo que no es clave depende de otro atributo que tampoco es clave, el cual a su vez depende de la clave primaria. Básicamente, toma su significado de la ley transitiva.

Fuente: https://www.datacamp.com/tutorial/normalization-in-sql


### Otras mejoras de diseño

Creación de tablas de referencia (lookup tables o reference tables), con base en los siguientes beneficios hipotéticos:

*   Optimización de Almacenamiento: la sustitución de texto repetitivo por un código numérico más ligero permite la reducción de espacio en disco, incrementa la velocidad de indexación y permite mayor agilidad en el filtrado.
*   Garantía de Integridad Referencial: Evitas que alguien escriba "Venta" en lugar de "Ventas" por error.
*   Gestión del Dominio de Datos. Al codificar los valores de cada categoría, se define formalmente el dominio del atributo en cuestión. Mediante una clave foránea, se puede forzar la integridad referencial, incrementando la consistencia de datos.
*   Flexibilidad. La utilización de tablas de referencia permite realizar modificaciones sobre los valores de las tablas de referencia, sin realizar modificaciones sobre las tablas de transacciones.

Sources: https://www.datacamp.com/es/tutorial/normalization-in-dbms

# Aspectos preliminares

In [2]:
# Importacion de librerias.
import pandas as pd
import numpy as np
import re
import os, sys, zipfile, shutil
from pandasql import sqldf

In [3]:
# Evaluacion del current directory y environment.
print(os.getcwd())
print(sys.executable)

d:\Github\data-architectures\01_data_normalization\notebooks
d:\Github\data-architectures\.venv\Scripts\python.exe


In [ ]:
# Directorio de datos.
data_path = '../../data/raw/'
output_dir = '../../db_input/'

# Application_test

### Analisis de datos preliminar

In [5]:
# Lectura del dataset.
csv_name = 'application_test.csv'
df = pd.read_csv(data_path + csv_name)

# Head del dataset.
df.info()

# Head del dataset.
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48744 entries, 0 to 48743
Columns: 121 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(40), object(16)
memory usage: 45.0+ MB


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Obtención de valores únicos para atributos categóricos y numéricos no float.
for column in df.columns:
    unique_values = df[column].unique()

    # Indicador para verificar si la columna contiene algún valor float no entero (excluyendo NaN)
    contains_non_integer_float = False

    # Iterar sobre los valores únicos para encontrar floats no enteros
    for val in unique_values:
        # Verificar si el valor es un float y no es NaN
        if isinstance(val, float) and not np.isnan(val):
            # Verificar si el float tiene una parte decimal distinta de cero
            if val != int(val):
                contains_non_integer_float = True
                break  # Se encontró un float no entero, no es necesario seguir verificando

    # Si no se encontró ningún float no entero, imprimir los valores únicos de la columna
    if not contains_non_integer_float:
        print(f"Columna '{column}':{unique_values}")

Columna 'SK_ID_CURR':[100001 100005 100013 ... 456223 456224 456250]
Columna 'NAME_CONTRACT_TYPE':['Cash loans' 'Revolving loans']
Columna 'CODE_GENDER':['F' 'M']
Columna 'FLAG_OWN_CAR':['N' 'Y']
Columna 'FLAG_OWN_REALTY':['Y' 'N']
Columna 'CNT_CHILDREN':[ 0  2  1  3  8  4  6  5  7 20 11]
Columna 'NAME_TYPE_SUITE':['Unaccompanied' nan 'Family' 'Spouse, partner' 'Group of people'
 'Other_B' 'Children' 'Other_A']
Columna 'NAME_INCOME_TYPE':['Working' 'State servant' 'Pensioner' 'Commercial associate'
 'Businessman' 'Student' 'Unemployed']
Columna 'NAME_EDUCATION_TYPE':['Higher education' 'Secondary / secondary special' 'Incomplete higher'
 'Lower secondary' 'Academic degree']
Columna 'NAME_FAMILY_STATUS':['Married' 'Single / not married' 'Civil marriage' 'Widow' 'Separated']
Columna 'NAME_HOUSING_TYPE':['House / apartment' 'With parents' 'Rented apartment'
 'Municipal apartment' 'Office apartment' 'Co-op apartment']
Columna 'DAYS_BIRTH':[-19241 -18064 -20038 ... -24168  -9192 -16019]
Col

In [7]:
# Obtención de valores únicos para atributos categóricos.
for column in df.columns:
    # Evaluación si el tipo es string.
    if df[column].dtype == 'object':
        unique_values = df[column].unique()
        print(f"Columna '{column}': {unique_values}")

Columna 'NAME_CONTRACT_TYPE': ['Cash loans' 'Revolving loans']
Columna 'CODE_GENDER': ['F' 'M']
Columna 'FLAG_OWN_CAR': ['N' 'Y']
Columna 'FLAG_OWN_REALTY': ['Y' 'N']
Columna 'NAME_TYPE_SUITE': ['Unaccompanied' nan 'Family' 'Spouse, partner' 'Group of people'
 'Other_B' 'Children' 'Other_A']
Columna 'NAME_INCOME_TYPE': ['Working' 'State servant' 'Pensioner' 'Commercial associate'
 'Businessman' 'Student' 'Unemployed']
Columna 'NAME_EDUCATION_TYPE': ['Higher education' 'Secondary / secondary special' 'Incomplete higher'
 'Lower secondary' 'Academic degree']
Columna 'NAME_FAMILY_STATUS': ['Married' 'Single / not married' 'Civil marriage' 'Widow' 'Separated']
Columna 'NAME_HOUSING_TYPE': ['House / apartment' 'With parents' 'Rented apartment'
 'Municipal apartment' 'Office apartment' 'Co-op apartment']
Columna 'OCCUPATION_TYPE': [nan 'Low-skill Laborers' 'Drivers' 'Sales staff' 'High skill tech staff'
 'Core staff' 'Laborers' 'Managers' 'Accountants' 'Medicine staff'
 'Security staff' 'Pri

### Primera forma normal

Del analisis realizado, se observó que el atributo "ORGANIZATION_TYPE" no contiene múltiples valores asociados entre sí. Por lo tanto, se incorpora una nueva columna para atomizar los valores.





In [8]:
# Creación de columna ORGANIZATION_TYPE_2 y atomización de valores.

def get_organization_type_2(org_type):
    if pd.isna(org_type):
        return 'Type 0' # Manejar valores NaN

    org_type_lower = str(org_type).lower()
    match_value = re.search(r'type\s*(\d+)', org_type_lower)
    if match_value:
        return f"Type {match_value.group(1)}"
    else:
        return "Type 0"

def clean_organization_type(org_type):
    if pd.isna(org_type):
        return org_type # Mantener valores NaN tal cual

    org_type_str = str(org_type)
    # Eliminar el patrón "type X", considerando dos puntos/espacios opcionales y sin importar mayúsculas/minúsculas
    cleaned_org_type = re.sub(r'[:-]?\s*type\s*\d+', '', org_type_str, flags=re.IGNORECASE).strip()
    # Si la cadena queda vacía después de limpiar (por ejemplo, si era solo "Type 1"), devolver la original
    return cleaned_org_type if cleaned_org_type else org_type_str

df['ORGANIZATION_TYPE_2'] = df['ORGANIZATION_TYPE'].apply(get_organization_type_2)
df['ORGANIZATION_TYPE'] = df['ORGANIZATION_TYPE'].apply(clean_organization_type)

In [9]:
# Print de valores únicos para verificar.
print(df['ORGANIZATION_TYPE'].value_counts(), end='\n\n')
print(df['ORGANIZATION_TYPE_2'].value_counts())

ORGANIZATION_TYPE
Business Entity        13206
XNA                     9274
Self-employed           5920
Other                   2707
Trade                   2332
Industry                2277
Medicine                1716
Transport               1541
Government              1508
School                  1287
Construction            1039
Kindergarten            1038
Military                 530
Security                 472
Police                   441
Housing                  435
Bank                     374
Security Ministries      341
Services                 302
Postal                   294
Agriculture              292
Restaurant               284
University               221
Electricity              156
Hotel                    134
Telecom                   95
Emergency                 91
Insurance                 80
Realtor                   72
Advertising               71
Culture                   61
Legal Services            53
Mobile                    45
Cleaning                 

### Segunda forma normal

No se detectaron dependencias parciales del análisis del dataset. Por lo tanto, no se realizaron modificaciones para transformar la base de acuerdo a la segunda forma normal.

### Tercera forma normal

La tercera forma normal requiere la eliminación de las dependencias transitivas entre atributos diferentes a la primary key. Del análisis no resultaron dependencias de esta naturaleza.

### Mejoras de diseño

In [10]:
# Creación de copia para control posterior.
df_backup = df.copy()

In [11]:
# Obtención de atributos categoricos.
dimension_tables = {}
categorical_columns = [col for col in df.columns if df[col].dtype == 'object']

In [12]:
# Creación de tablas de dimensiones.
for col in categorical_columns:
    # Extraer valores únicos no NaN para la tabla de dimensión
    unique_values = df[col].dropna().unique()

    if len(unique_values) > 0:
        # Crear una tabla de dimensión para la columna categórica actual
        dim_df = pd.DataFrame({
            f'{col}_ID': range(1, len(unique_values) + 1),
            col: unique_values
        })
        dimension_tables[f'dim_{col.lower()}'] = dim_df

        # Crear un mapeo desde el valor original al nuevo ID, manejando NaN
        mapping = dict(zip(dim_df[col], dim_df[f'{col}_ID']))

        # Reemplazar los valores categóricos en el DataFrame original con sus IDs
        # Usar .map() que maneja valores NaN por defecto, preservándolos si no están en el mapeo
        df[col] = df[col].map(mapping)

        # Renombrar la columna para reflejar que ahora contiene IDs
        df.rename(columns={col: f'{col}_ID'}, inplace=True)
    else:
        print(f"La columna '{col}' no tiene valores únicos que no estén vacios, por lo tanto se omite la creación de la tabla de dimensión.")

In [13]:
# Control de atributos.
print(f'Recuento de columnas categoricas: {len(categorical_columns)}')
print(f'Recuento de tablas de dimensiones: {len(dimension_tables)}')

Recuento de columnas categoricas: 17
Recuento de tablas de dimensiones: 17


In [14]:
# Tablas de dimensiones.
print(f'Tablas de dimensiones: {[*dimension_tables.keys()]}')

Tablas de dimensiones: ['dim_name_contract_type', 'dim_code_gender', 'dim_flag_own_car', 'dim_flag_own_realty', 'dim_name_type_suite', 'dim_name_income_type', 'dim_name_education_type', 'dim_name_family_status', 'dim_name_housing_type', 'dim_occupation_type', 'dim_weekday_appr_process_start', 'dim_organization_type', 'dim_fondkapremont_mode', 'dim_housetype_mode', 'dim_wallsmaterial_mode', 'dim_emergencystate_mode', 'dim_organization_type_2']


In [15]:
# Print de atributos del dataset backup.
print(f'Columnas del dataset de referencia: {[*df_backup.columns]}')

Columnas del dataset de referencia: ['SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'ORGANIZATION_TYPE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AV

In [16]:
# Query para la verificación del join de tablas principal y de dimensiones.
query = f"""
SELECT {', '.join(df_backup.columns)}
FROM df
"""

dim_tables_names = [*dimension_tables.keys()]

for dim_table_name in dim_tables_names:
  dim_table_id = dimension_tables[dim_table_name].columns[0]

  query += f"""
  LEFT JOIN {dim_table_name} ON df.{dim_table_id} = {dim_table_name}.{dim_table_id}
  """

In [17]:
# Impresion de la query.
print(query)


SELECT SK_ID_CURR, NAME_CONTRACT_TYPE, CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY, CNT_CHILDREN, AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, AMT_GOODS_PRICE, NAME_TYPE_SUITE, NAME_INCOME_TYPE, NAME_EDUCATION_TYPE, NAME_FAMILY_STATUS, NAME_HOUSING_TYPE, REGION_POPULATION_RELATIVE, DAYS_BIRTH, DAYS_EMPLOYED, DAYS_REGISTRATION, DAYS_ID_PUBLISH, OWN_CAR_AGE, FLAG_MOBIL, FLAG_EMP_PHONE, FLAG_WORK_PHONE, FLAG_CONT_MOBILE, FLAG_PHONE, FLAG_EMAIL, OCCUPATION_TYPE, CNT_FAM_MEMBERS, REGION_RATING_CLIENT, REGION_RATING_CLIENT_W_CITY, WEEKDAY_APPR_PROCESS_START, HOUR_APPR_PROCESS_START, REG_REGION_NOT_LIVE_REGION, REG_REGION_NOT_WORK_REGION, LIVE_REGION_NOT_WORK_REGION, REG_CITY_NOT_LIVE_CITY, REG_CITY_NOT_WORK_CITY, LIVE_CITY_NOT_WORK_CITY, ORGANIZATION_TYPE, EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3, APARTMENTS_AVG, BASEMENTAREA_AVG, YEARS_BEGINEXPLUATATION_AVG, YEARS_BUILD_AVG, COMMONAREA_AVG, ELEVATORS_AVG, ENTRANCES_AVG, FLOORSMAX_AVG, FLOORSMIN_AVG, LANDAREA_AVG, LIVINGAPARTMENTS_AVG, LIVINGA

In [18]:
# Control de join de tablas descompuestas.

# Construcción de diccionario con dataframe principal y dataframes de dimensiones.
env = {**dimension_tables, 'df': df}

# Ejecución de la query.
control_df = sqldf(query, env) # sqldf(query, locals())
control_df.head()

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,ORGANIZATION_TYPE_2
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,Type 0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0,Type 0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0,Type 3
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0,Type 3
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Type 3


In [19]:
# Shape de control_df.
control_df.shape

(48744, 122)

In [20]:
# Reemplazo de NoneType por np.nans.
for col in control_df.columns:
    # Verificacion si 'None' (NoneType) está presente en la columna.
    if control_df[col].apply(lambda x: x is None).any():
        control_df[col] = control_df[col].replace({None: np.nan})

print("Valores None fueron reemplazados con np.nan para todas las columnas de control_df.")

Valores None fueron reemplazados con np.nan para todas las columnas de control_df.


In [21]:
# Cotejo final de control_df con df_backup.

matching_counts = {}

# Asegurar que solo comparamos las columnas que existen en ambos DataFrames.
columns_to_compare = [col for col in df_backup.columns if col in control_df.columns]

for col in columns_to_compare:
    s1 = control_df[col]
    s2 = df_backup[col]

    # Comparar valores, tratando np.nan como iguales si ambos lo son
    # Esto gestiona tanto valores iguales no nulos (s1 == s2) como coincidencias NaN-NaN (s1.isna() & s2.isna())
    matches = ((s1 == s2) | (s1.isna() & s2.isna()))
    matching_counts[col] = matches.sum()

# Crear un DataFrame para visualizar mejor los resultados
comparison_df = pd.DataFrame(matching_counts.items(), columns=['Column', 'Matching_Count'])
comparison_df['Total_Rows'] = len(df_backup)
comparison_df['Percentage_Match'] = (comparison_df['Matching_Count'] / comparison_df['Total_Rows']) * 100

display(comparison_df)

# Opcionalmente, mostrar columnas que no tienen un 100% de coincidencia para facilitar la inspección
non_full_match_cols = comparison_df[comparison_df['Percentage_Match'] < 100]
if not non_full_match_cols.empty:
    print("\nColumnas con menos del 100% de coincidencia (debido a diferencias reales de valores, no solo manejo de NaN):")
    display(non_full_match_cols)
else:
    print("\nTodas las columnas comparadas tienen una coincidencia del 100% (incluyendo la igualdad NaN-NaN).")

,Column,Matching_Count,Total_Rows,Percentage_Match
0,SK_ID_CURR,48744,48744,100.0
1,NAME_CONTRACT_TYPE,48744,48744,100.0
2,CODE_GENDER,48744,48744,100.0
3,FLAG_OWN_CAR,48744,48744,100.0
4,FLAG_OWN_REALTY,48744,48744,100.0
...,...,...,...,...
117,AMT_REQ_CREDIT_BUREAU_WEEK,48744,48744,100.0
118,AMT_REQ_CREDIT_BUREAU_MON,48744,48744,100.0
119,AMT_REQ_CREDIT_BUREAU_QRT,48744,48744,100.0
120,AMT_REQ_CREDIT_BUREAU_YEAR,48744,48744,100.0



Todas las columnas comparadas tienen una coincidencia del 100% (incluyendo la igualdad NaN-NaN).


### Exportación de DataFrames a CSV

Este paso guarda el DataFrame principal y cada una de las tablas de dimensiones creadas en archivos CSV individuales. Esto permite un almacenamiento y uso posterior más sencillo de los datos normalizados y las tablas de referencia.

In [25]:
# Creacion de directorio output.
ref_name = csv_name.replace('.csv', '')
extract_dir = output_dir + ref_name + '/'

# Evaluacion de existencia del directorio.
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir)

# Guardar el dataFrame principal 'df' en un archivo CSV.
df.to_csv(extract_dir + csv_name, index=False)
print(f'DataFrame principal guardado como {csv_name}')

# Guardar cada tabla de dimensiones en un archivo CSV separado
for name, dim_df in dimension_tables.items():
    file_name = f'{name}.csv'
    dim_df.to_csv(extract_dir + file_name, index=False)
    print(f'Tabla de dimensiones "{name}" guardada como {file_name}')

DataFrame principal guardado como application_test.csv
Tabla de dimensiones "dim_name_contract_type" guardada como dim_name_contract_type.csv
Tabla de dimensiones "dim_code_gender" guardada como dim_code_gender.csv
Tabla de dimensiones "dim_flag_own_car" guardada como dim_flag_own_car.csv
Tabla de dimensiones "dim_flag_own_realty" guardada como dim_flag_own_realty.csv
Tabla de dimensiones "dim_name_type_suite" guardada como dim_name_type_suite.csv
Tabla de dimensiones "dim_name_income_type" guardada como dim_name_income_type.csv
Tabla de dimensiones "dim_name_education_type" guardada como dim_name_education_type.csv
Tabla de dimensiones "dim_name_family_status" guardada como dim_name_family_status.csv
Tabla de dimensiones "dim_name_housing_type" guardada como dim_name_housing_type.csv
Tabla de dimensiones "dim_occupation_type" guardada como dim_occupation_type.csv
Tabla de dimensiones "dim_weekday_appr_process_start" guardada como dim_weekday_appr_process_start.csv
Tabla de dimensiones

# Application_train

### Analisis de datos preliminar





In [26]:
# Lectura del dataset.
csv_name = 'application_train.csv'
df = pd.read_csv(data_path + csv_name)

# Head del dataset.
df.info()

# Head del dataset.
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), object(16)
memory usage: 286.2+ MB


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
# Obtención de valores únicos para atributos categóricos y numéricos no float.
for column in df.columns:
    unique_values = df[column].unique()

    # Indicador para verificar si la columna contiene algún valor float no entero (excluyendo NaN)
    contains_non_integer_float = False

    # Iterar sobre los valores únicos para encontrar floats no enteros
    for val in unique_values:
        # Verificar si el valor es un float y no es NaN
        if isinstance(val, float) and not np.isnan(val):
            # Verificar si el float tiene una parte decimal distinta de cero
            if val != int(val):
                contains_non_integer_float = True
                break  # Se encontró un float no entero, no es necesario seguir verificando

    # Si no se encontró ningún float no entero, imprimir los valores únicos de la columna
    if not contains_non_integer_float:
        print(f"Columna '{column}':{unique_values}")

Columna 'SK_ID_CURR':[100002 100003 100004 ... 456253 456254 456255]
Columna 'TARGET':[1 0]
Columna 'NAME_CONTRACT_TYPE':['Cash loans' 'Revolving loans']
Columna 'CODE_GENDER':['M' 'F' 'XNA']
Columna 'FLAG_OWN_CAR':['N' 'Y']
Columna 'FLAG_OWN_REALTY':['Y' 'N']
Columna 'CNT_CHILDREN':[ 0  1  2  3  4  7  5  6  8  9 11 12 10 19 14]
Columna 'NAME_TYPE_SUITE':['Unaccompanied' 'Family' 'Spouse, partner' 'Children' 'Other_A' nan
 'Other_B' 'Group of people']
Columna 'NAME_INCOME_TYPE':['Working' 'State servant' 'Commercial associate' 'Pensioner' 'Unemployed'
 'Student' 'Businessman' 'Maternity leave']
Columna 'NAME_EDUCATION_TYPE':['Secondary / secondary special' 'Higher education' 'Incomplete higher'
 'Lower secondary' 'Academic degree']
Columna 'NAME_FAMILY_STATUS':['Single / not married' 'Married' 'Civil marriage' 'Widow' 'Separated'
 'Unknown']
Columna 'NAME_HOUSING_TYPE':['House / apartment' 'Rented apartment' 'With parents'
 'Municipal apartment' 'Office apartment' 'Co-op apartment']
Co

In [28]:
# Obtención de valores únicos para atributos categóricos.
for column in df.columns:
    # Evaluación si el tipo es string.
    if df[column].dtype == 'object':
        unique_values = df[column].unique()
        print(f"Columna '{column}': {unique_values}")

Columna 'NAME_CONTRACT_TYPE': ['Cash loans' 'Revolving loans']
Columna 'CODE_GENDER': ['M' 'F' 'XNA']
Columna 'FLAG_OWN_CAR': ['N' 'Y']
Columna 'FLAG_OWN_REALTY': ['Y' 'N']
Columna 'NAME_TYPE_SUITE': ['Unaccompanied' 'Family' 'Spouse, partner' 'Children' 'Other_A' nan
 'Other_B' 'Group of people']
Columna 'NAME_INCOME_TYPE': ['Working' 'State servant' 'Commercial associate' 'Pensioner' 'Unemployed'
 'Student' 'Businessman' 'Maternity leave']
Columna 'NAME_EDUCATION_TYPE': ['Secondary / secondary special' 'Higher education' 'Incomplete higher'
 'Lower secondary' 'Academic degree']
Columna 'NAME_FAMILY_STATUS': ['Single / not married' 'Married' 'Civil marriage' 'Widow' 'Separated'
 'Unknown']
Columna 'NAME_HOUSING_TYPE': ['House / apartment' 'Rented apartment' 'With parents'
 'Municipal apartment' 'Office apartment' 'Co-op apartment']
Columna 'OCCUPATION_TYPE': ['Laborers' 'Core staff' 'Accountants' 'Managers' nan 'Drivers'
 'Sales staff' 'Cleaning staff' 'Cooking staff' 'Private service

### Primera forma normal

Del analisis realizado, se observó que el atributo "ORGANIZATION_TYPE" no contiene múltiples valores asociados entre sí. Por lo tanto, se incorpora una nueva columna para atomizar los valores.





In [29]:
# Creación de columna ORGANIZATION_TYPE_2 y atomización de valores.

def get_organization_type_2(org_type):
    if pd.isna(org_type):
        return 'Type 0' # Manejar valores NaN

    org_type_lower = str(org_type).lower()
    match_value = re.search(r'type\s*(\d+)', org_type_lower)
    if match_value:
        return f"Type {match_value.group(1)}"
    else:
        return "Type 0"

def clean_organization_type(org_type):
    if pd.isna(org_type):
        return org_type # Mantener valores NaN tal cual

    org_type_str = str(org_type)
    # Eliminar el patrón "type X", considerando dos puntos/espacios opcionales y sin importar mayúsculas/minúsculas
    cleaned_org_type = re.sub(r'[:-]?\s*type\s*\d+', '', org_type_str, flags=re.IGNORECASE).strip()
    # Si la cadena queda vacía después de limpiar (por ejemplo, si era solo "Type 1"), devolver la original
    return cleaned_org_type if cleaned_org_type else org_type_str

df['ORGANIZATION_TYPE_2'] = df['ORGANIZATION_TYPE'].apply(get_organization_type_2)
df['ORGANIZATION_TYPE'] = df['ORGANIZATION_TYPE'].apply(clean_organization_type)

In [30]:
# Print de valores únicos para verificar.
print(df['ORGANIZATION_TYPE'].value_counts(), end='\n\n')
print(df['ORGANIZATION_TYPE_2'].value_counts())

ORGANIZATION_TYPE
Business Entity        84529
XNA                    55374
Self-employed          38412
Other                  16683
Trade                  14315
Industry               14311
Medicine               11193
Government             10404
Transport               8990
School                  8893
Kindergarten            6880
Construction            6721
Security                3247
Housing                 2958
Military                2634
Bank                    2507
Agriculture             2454
Police                  2341
Postal                  2157
Security Ministries     1974
Restaurant              1811
Services                1575
University              1327
Hotel                    966
Electricity              950
Insurance                597
Telecom                  577
Emergency                560
Advertising              429
Realtor                  396
Culture                  379
Mobile                   317
Legal Services           305
Cleaning                 

### Segunda forma normal

No se detectaron dependencias parciales del análisis del dataset. Por lo tanto, no se realizaron modificaciones para transformar la base de acuerdo a la segunda forma normal.

### Tercera forma normal

La tercera forma normal requiere la eliminación de las dependencias transitivas entre atributos diferentes a la primary key. Del análisis no resultaron dependencias de esta naturaleza.

### Mejoras de diseño

In [31]:
# Creación de copia para control posterior.
df_backup = df.copy()

In [32]:
# Obtención de atributos categoricos.
dimension_tables = {}
categorical_columns = [col for col in df.columns if df[col].dtype == 'object']

In [33]:
# Creación de tablas de dimensiones.
for col in categorical_columns:
    # Extraer valores únicos no NaN para la tabla de dimensión
    unique_values = df[col].dropna().unique()

    if len(unique_values) > 0:
        # Crear una tabla de dimensión para la columna categórica actual
        dim_df = pd.DataFrame({
            f'{col}_ID': range(1, len(unique_values) + 1),
            col: unique_values
        })
        dimension_tables[f'dim_{col.lower()}'] = dim_df

        # Crear un mapeo desde el valor original al nuevo ID, manejando NaN
        mapping = dict(zip(dim_df[col], dim_df[f'{col}_ID']))

        # Reemplazar los valores categóricos en el DataFrame original con sus IDs
        # Usar .map() que maneja valores NaN por defecto, preservándolos si no están en el mapeo
        df[col] = df[col].map(mapping)

        # Renombrar la columna para reflejar que ahora contiene IDs
        df.rename(columns={col: f'{col}_ID'}, inplace=True)
    else:
        print(f"La columna '{col}' no tiene valores únicos que no estén vacios, por lo tanto se omite la creación de la tabla de dimensión.")

In [34]:
# Control de atributos.
print(f'Recuento de columnas categoricas: {len(categorical_columns)}')
print(f'Recuento de tablas de dimensiones: {len(dimension_tables)}')

Recuento de columnas categoricas: 17
Recuento de tablas de dimensiones: 17


In [35]:
# Tablas de dimensiones.
print(f'Tablas de dimensiones: {[*dimension_tables.keys()]}')

Tablas de dimensiones: ['dim_name_contract_type', 'dim_code_gender', 'dim_flag_own_car', 'dim_flag_own_realty', 'dim_name_type_suite', 'dim_name_income_type', 'dim_name_education_type', 'dim_name_family_status', 'dim_name_housing_type', 'dim_occupation_type', 'dim_weekday_appr_process_start', 'dim_organization_type', 'dim_fondkapremont_mode', 'dim_housetype_mode', 'dim_wallsmaterial_mode', 'dim_emergencystate_mode', 'dim_organization_type_2']


In [36]:
# Print de atributos del dataset backup.
print(f'Columnas del dataset de referencia: {[*df_backup.columns]}')

Columnas del dataset de referencia: ['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'ORGANIZATION_TYPE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEAR

In [37]:
# Query para la verificación del join de tablas principal y de dimensiones.
query = f"""
SELECT {', '.join(df_backup.columns)}
FROM df
"""

dim_tables_names = [*dimension_tables.keys()]

for dim_table_name in dim_tables_names:
  dim_table_id = dimension_tables[dim_table_name].columns[0]

  query += f"""
  LEFT JOIN {dim_table_name} ON df.{dim_table_id} = {dim_table_name}.{dim_table_id}
  """

In [38]:
# Impresion de la query.
print(query)


SELECT SK_ID_CURR, TARGET, NAME_CONTRACT_TYPE, CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY, CNT_CHILDREN, AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, AMT_GOODS_PRICE, NAME_TYPE_SUITE, NAME_INCOME_TYPE, NAME_EDUCATION_TYPE, NAME_FAMILY_STATUS, NAME_HOUSING_TYPE, REGION_POPULATION_RELATIVE, DAYS_BIRTH, DAYS_EMPLOYED, DAYS_REGISTRATION, DAYS_ID_PUBLISH, OWN_CAR_AGE, FLAG_MOBIL, FLAG_EMP_PHONE, FLAG_WORK_PHONE, FLAG_CONT_MOBILE, FLAG_PHONE, FLAG_EMAIL, OCCUPATION_TYPE, CNT_FAM_MEMBERS, REGION_RATING_CLIENT, REGION_RATING_CLIENT_W_CITY, WEEKDAY_APPR_PROCESS_START, HOUR_APPR_PROCESS_START, REG_REGION_NOT_LIVE_REGION, REG_REGION_NOT_WORK_REGION, LIVE_REGION_NOT_WORK_REGION, REG_CITY_NOT_LIVE_CITY, REG_CITY_NOT_WORK_CITY, LIVE_CITY_NOT_WORK_CITY, ORGANIZATION_TYPE, EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3, APARTMENTS_AVG, BASEMENTAREA_AVG, YEARS_BEGINEXPLUATATION_AVG, YEARS_BUILD_AVG, COMMONAREA_AVG, ELEVATORS_AVG, ENTRANCES_AVG, FLOORSMAX_AVG, FLOORSMIN_AVG, LANDAREA_AVG, LIVINGAPARTMENTS_AVG,

In [39]:
# Control de join de tablas descompuestas.

# Construcción de diccionario con dataframe principal y dataframes de dimensiones.
env = {**dimension_tables, 'df': df}

# Ejecución de la query.
control_df = sqldf(query, env) # sqldf(query, locals())
control_df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,ORGANIZATION_TYPE_2
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,Type 3
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,Type 0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,Type 0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Type 3
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,Type 0


In [40]:
# Shape de control_df.
control_df.shape

(307511, 123)

In [41]:
# Reemplazo de NoneType por np.nans.
for col in control_df.columns:
    # Verificacion si 'None' (NoneType) está presente en la columna.
    if control_df[col].apply(lambda x: x is None).any():
        control_df[col] = control_df[col].replace({None: np.nan})

print("Valores None fueron reemplazados con np.nan para todas las columnas de control_df.")

Valores None fueron reemplazados con np.nan para todas las columnas de control_df.


In [42]:
# Cotejo final de control_df con df_backup.

matching_counts = {}

# Asegurar que solo comparamos las columnas que existen en ambos DataFrames.
columns_to_compare = [col for col in df_backup.columns if col in control_df.columns]

for col in columns_to_compare:
    s1 = control_df[col]
    s2 = df_backup[col]

    # Comparar valores, tratando np.nan como iguales si ambos lo son
    # Esto gestiona tanto valores iguales no nulos (s1 == s2) como coincidencias NaN-NaN (s1.isna() & s2.isna())
    matches = ((s1 == s2) | (s1.isna() & s2.isna()))
    matching_counts[col] = matches.sum()

# Crear un DataFrame para visualizar mejor los resultados
comparison_df = pd.DataFrame(matching_counts.items(), columns=['Column', 'Matching_Count'])
comparison_df['Total_Rows'] = len(df_backup)
comparison_df['Percentage_Match'] = (comparison_df['Matching_Count'] / comparison_df['Total_Rows']) * 100

display(comparison_df)

# Opcionalmente, mostrar columnas que no tienen un 100% de coincidencia para facilitar la inspección
non_full_match_cols = comparison_df[comparison_df['Percentage_Match'] < 100]
if not non_full_match_cols.empty:
    print("\nColumnas con menos del 100% de coincidencia (debido a diferencias reales de valores, no solo manejo de NaN):")
    display(non_full_match_cols)
else:
    print("\nTodas las columnas comparadas tienen una coincidencia del 100% (incluyendo la igualdad NaN-NaN).")

,Column,Matching_Count,Total_Rows,Percentage_Match
0,SK_ID_CURR,307511,307511,100.0
1,TARGET,307511,307511,100.0
2,NAME_CONTRACT_TYPE,307511,307511,100.0
3,CODE_GENDER,307511,307511,100.0
4,FLAG_OWN_CAR,307511,307511,100.0
...,...,...,...,...
118,AMT_REQ_CREDIT_BUREAU_WEEK,307511,307511,100.0
119,AMT_REQ_CREDIT_BUREAU_MON,307511,307511,100.0
120,AMT_REQ_CREDIT_BUREAU_QRT,307511,307511,100.0
121,AMT_REQ_CREDIT_BUREAU_YEAR,307511,307511,100.0



Todas las columnas comparadas tienen una coincidencia del 100% (incluyendo la igualdad NaN-NaN).


### Exportación de DataFrames a CSV

Este paso guarda el DataFrame principal y cada una de las tablas de dimensiones creadas en archivos CSV individuales. Esto permite un almacenamiento y uso posterior más sencillo de los datos normalizados y las tablas de referencia.

In [43]:
# Creacion de directorio output.
ref_name = csv_name.replace('.csv', '')
extract_dir = output_dir + ref_name + '/'

# Evaluacion de existencia del directorio.
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir)

# Guardar el dataFrame principal 'df' en un archivo CSV.
df.to_csv(extract_dir + csv_name, index=False)
print(f'DataFrame principal guardado como {csv_name}')

# Guardar cada tabla de dimensiones en un archivo CSV separado
for name, dim_df in dimension_tables.items():
    file_name = f'{name}.csv'
    dim_df.to_csv(extract_dir + file_name, index=False)
    print(f'Tabla de dimensiones "{name}" guardada como {file_name}')

DataFrame principal guardado como application_train.csv
Tabla de dimensiones "dim_name_contract_type" guardada como dim_name_contract_type.csv
Tabla de dimensiones "dim_code_gender" guardada como dim_code_gender.csv
Tabla de dimensiones "dim_flag_own_car" guardada como dim_flag_own_car.csv
Tabla de dimensiones "dim_flag_own_realty" guardada como dim_flag_own_realty.csv
Tabla de dimensiones "dim_name_type_suite" guardada como dim_name_type_suite.csv
Tabla de dimensiones "dim_name_income_type" guardada como dim_name_income_type.csv
Tabla de dimensiones "dim_name_education_type" guardada como dim_name_education_type.csv
Tabla de dimensiones "dim_name_family_status" guardada como dim_name_family_status.csv
Tabla de dimensiones "dim_name_housing_type" guardada como dim_name_housing_type.csv
Tabla de dimensiones "dim_occupation_type" guardada como dim_occupation_type.csv
Tabla de dimensiones "dim_weekday_appr_process_start" guardada como dim_weekday_appr_process_start.csv
Tabla de dimensione

# bureau_balance

### Analisis de datos preliminar





In [44]:
# Lectura del dataset.
csv_name = 'bureau_balance.csv'
df = pd.read_csv(data_path + csv_name)

# Head del dataset.
df.info()

# Head del dataset.
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27299925 entries, 0 to 27299924
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   SK_ID_BUREAU    int64 
 1   MONTHS_BALANCE  int64 
 2   STATUS          object
dtypes: int64(2), object(1)
memory usage: 624.8+ MB


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [45]:
# Obtención de valores únicos para atributos categóricos y numéricos no float.
for column in df.columns:
    unique_values = df[column].unique()

    # Indicador para verificar si la columna contiene algún valor float no entero (excluyendo NaN)
    contains_non_integer_float = False

    # Iterar sobre los valores únicos para encontrar floats no enteros
    for val in unique_values:
        # Verificar si el valor es un float y no es NaN
        if isinstance(val, float) and not np.isnan(val):
            # Verificar si el float tiene una parte decimal distinta de cero
            if val != int(val):
                contains_non_integer_float = True
                break  # Se encontró un float no entero, no es necesario seguir verificando

    # Si no se encontró ningún float no entero, imprimir los valores únicos de la columna
    if not contains_non_integer_float:
        print(f"Columna '{column}':{unique_values}")

Columna 'SK_ID_BUREAU':[5715448 5715449 5715451 ... 5041172 5041332 5041336]
Columna 'MONTHS_BALANCE':[  0  -1  -2  -3  -4  -5  -6  -7  -8  -9 -10 -11 -12 -13 -14 -15 -16 -17
 -18 -19 -20 -21 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31 -32 -33 -34 -35
 -36 -37 -38 -39 -40 -41 -42 -43 -44 -45 -46 -47 -48 -49 -50 -51 -52 -53
 -54 -55 -56 -57 -58 -59 -60 -61 -62 -63 -64 -65 -66 -67 -68 -69 -70 -71
 -72 -73 -74 -75 -76 -77 -78 -79 -80 -81 -82 -83 -84 -85 -86 -87 -88 -89
 -90 -91 -92 -93 -94 -95 -96]
Columna 'STATUS':['C' '0' 'X' '1' '2' '3' '5' '4']


In [46]:
# Obtención de valores únicos para atributos categóricos.
for column in df.columns:
    # Evaluación si el tipo es string.
    if df[column].dtype == 'object':
        unique_values = df[column].unique()
        print(f"Columna '{column}': {unique_values}")

Columna 'STATUS': ['C' '0' 'X' '1' '2' '3' '5' '4']


### Mejoras de diseño

Dado que un sólo atributo es optimizable, se mantienen los datos en la tabla en el mismo estado en que se encuentran.



# bureau

### Analisis de datos preliminar





In [47]:
# Lectura del dataset.
csv_name = 'bureau.csv'
df = pd.read_csv(data_path + csv_name)

# Head del dataset.
df.info()

# Head del dataset.
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_CURR              int64  
 1   SK_ID_BUREAU            int64  
 2   CREDIT_ACTIVE           object 
 3   CREDIT_CURRENCY         object 
 4   DAYS_CREDIT             int64  
 5   CREDIT_DAY_OVERDUE      int64  
 6   DAYS_CREDIT_ENDDATE     float64
 7   DAYS_ENDDATE_FACT       float64
 8   AMT_CREDIT_MAX_OVERDUE  float64
 9   CNT_CREDIT_PROLONG      int64  
 10  AMT_CREDIT_SUM          float64
 11  AMT_CREDIT_SUM_DEBT     float64
 12  AMT_CREDIT_SUM_LIMIT    float64
 13  AMT_CREDIT_SUM_OVERDUE  float64
 14  CREDIT_TYPE             object 
 15  DAYS_CREDIT_UPDATE      int64  
 16  AMT_ANNUITY             float64
dtypes: float64(8), int64(6), object(3)
memory usage: 222.6+ MB


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [48]:
# Obtención de valores únicos para atributos categóricos y numéricos no float.
for column in df.columns:
    unique_values = df[column].unique()

    # Indicador para verificar si la columna contiene algún valor float no entero (excluyendo NaN)
    contains_non_integer_float = False

    # Iterar sobre los valores únicos para encontrar floats no enteros
    for val in unique_values:
        # Verificar si el valor es un float y no es NaN
        if isinstance(val, float) and not np.isnan(val):
            # Verificar si el float tiene una parte decimal distinta de cero
            if val != int(val):
                contains_non_integer_float = True
                break  # Se encontró un float no entero, no es necesario seguir verificando

    # Si no se encontró ningún float no entero, imprimir los valores únicos de la columna
    if not contains_non_integer_float:
        print(f"Columna '{column}':{unique_values}")

Columna 'SK_ID_CURR':[215354 162297 402440 ... 448157 345866 235871]
Columna 'SK_ID_BUREAU':[5714462 5714463 5714464 ... 5057762 5057770 5057778]
Columna 'CREDIT_ACTIVE':['Closed' 'Active' 'Sold' 'Bad debt']
Columna 'CREDIT_CURRENCY':['currency 1' 'currency 2' 'currency 4' 'currency 3']
Columna 'DAYS_CREDIT':[-497 -208 -203 ...    0   -3   -1]
Columna 'CREDIT_DAY_OVERDUE':[   0 2603    6   30 2156  496  186 2264   41    8   71   13  252 1201
   12   38 1777  150   18   37   64 2050    9    4   29   22   49 1930
   34  951  936  664   45 2625  837   21   15   31 1715   19   67    7
   26   23   48   46   73   90 1127   53   33 1572   39 2582   14   25
   10   50   17  193   40   20   28  123  138  700  122 1504   32  670
   75  140   24  277 2019  161  962  187   58  106  634   83  120   60
   93 1694  227   44  344 2085 1248  124 1651 2541 1541  642  745  163
  234 1472  688 2455  614  319 2187   68  831  139  283  180   47  164
  174 1604 1326 2149   36   62  753  374 1573 1562 2400  

In [49]:
# Obtención de valores únicos para atributos categóricos.
for column in df.columns:
    # Evaluación si el tipo es string.
    if df[column].dtype == 'object':
        unique_values = df[column].unique()
        print(f"Columna '{column}': {unique_values}")

Columna 'CREDIT_ACTIVE': ['Closed' 'Active' 'Sold' 'Bad debt']
Columna 'CREDIT_CURRENCY': ['currency 1' 'currency 2' 'currency 4' 'currency 3']
Columna 'CREDIT_TYPE': ['Consumer credit' 'Credit card' 'Mortgage' 'Car loan' 'Microloan'
 'Loan for working capital replenishment' 'Loan for business development'
 'Real estate loan' 'Unknown type of loan' 'Another type of loan'
 'Cash loan (non-earmarked)' 'Loan for the purchase of equipment'
 'Mobile operator loan' 'Interbank credit'
 'Loan for purchase of shares (margin lending)']


### Mejoras de diseño

In [50]:
# Creación de copia para control posterior.
df_backup = df.copy()

In [51]:
# Obtención de atributos categoricos.
dimension_tables = {}
categorical_columns = [col for col in df.columns if df[col].dtype == 'object']

In [52]:
# Creación de tablas de dimensiones.
for col in categorical_columns:
    # Extraer valores únicos no NaN para la tabla de dimensión
    unique_values = df[col].dropna().unique()

    if len(unique_values) > 0:
        # Crear una tabla de dimensión para la columna categórica actual
        dim_df = pd.DataFrame({
            f'{col}_ID': range(1, len(unique_values) + 1),
            col: unique_values
        })
        dimension_tables[f'dim_{col.lower()}'] = dim_df

        # Crear un mapeo desde el valor original al nuevo ID, manejando NaN
        mapping = dict(zip(dim_df[col], dim_df[f'{col}_ID']))

        # Reemplazar los valores categóricos en el DataFrame original con sus IDs
        # Usar .map() que maneja valores NaN por defecto, preservándolos si no están en el mapeo
        df[col] = df[col].map(mapping)

        # Renombrar la columna para reflejar que ahora contiene IDs
        df.rename(columns={col: f'{col}_ID'}, inplace=True)
    else:
        print(f"La columna '{col}' no tiene valores únicos que no estén vacios, por lo tanto se omite la creación de la tabla de dimensión.")

In [53]:
# Control de atributos.
print(f'Recuento de columnas categoricas: {len(categorical_columns)}')
print(f'Recuento de tablas de dimensiones: {len(dimension_tables)}')

Recuento de columnas categoricas: 3
Recuento de tablas de dimensiones: 3


In [54]:
# Tablas de dimensiones.
print(f'Tablas de dimensiones: {[*dimension_tables.keys()]}')

Tablas de dimensiones: ['dim_credit_active', 'dim_credit_currency', 'dim_credit_type']


In [55]:
# Print de atributos del dataset backup.
print(f'Columnas del dataset de referencia: {[*df_backup.columns]}')

Columnas del dataset de referencia: ['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']


In [56]:
# Query para la verificación del join de tablas principal y de dimensiones.
query = f"""
SELECT {', '.join(df_backup.columns)}
FROM df
"""

dim_tables_names = [*dimension_tables.keys()]

for dim_table_name in dim_tables_names:
  dim_table_id = dimension_tables[dim_table_name].columns[0]

  query += f"""
  LEFT JOIN {dim_table_name} ON df.{dim_table_id} = {dim_table_name}.{dim_table_id}
  """

In [57]:
# Impresion de la query.
print(query)


SELECT SK_ID_CURR, SK_ID_BUREAU, CREDIT_ACTIVE, CREDIT_CURRENCY, DAYS_CREDIT, CREDIT_DAY_OVERDUE, DAYS_CREDIT_ENDDATE, DAYS_ENDDATE_FACT, AMT_CREDIT_MAX_OVERDUE, CNT_CREDIT_PROLONG, AMT_CREDIT_SUM, AMT_CREDIT_SUM_DEBT, AMT_CREDIT_SUM_LIMIT, AMT_CREDIT_SUM_OVERDUE, CREDIT_TYPE, DAYS_CREDIT_UPDATE, AMT_ANNUITY
FROM df

  LEFT JOIN dim_credit_active ON df.CREDIT_ACTIVE_ID = dim_credit_active.CREDIT_ACTIVE_ID
  
  LEFT JOIN dim_credit_currency ON df.CREDIT_CURRENCY_ID = dim_credit_currency.CREDIT_CURRENCY_ID
  
  LEFT JOIN dim_credit_type ON df.CREDIT_TYPE_ID = dim_credit_type.CREDIT_TYPE_ID
  


In [58]:
# Control de join de tablas descompuestas.

# Construcción de diccionario con dataframe principal y dataframes de dimensiones.
env = {**dimension_tables, 'df': df}

# Ejecución de la query.
control_df = sqldf(query, env) # sqldf(query, locals())
control_df.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [59]:
# Shape de control_df.
control_df.shape

(1716428, 17)

In [60]:
# Reemplazo de NoneType por np.nans.
for col in control_df.columns:
    # Verificacion si 'None' (NoneType) está presente en la columna.
    if control_df[col].apply(lambda x: x is None).any():
        control_df[col] = control_df[col].replace({None: np.nan})

print("Valores None fueron reemplazados con np.nan para todas las columnas de control_df.")

Valores None fueron reemplazados con np.nan para todas las columnas de control_df.


In [61]:
# Cotejo final de control_df con df_backup.

matching_counts = {}

# Asegurar que solo comparamos las columnas que existen en ambos DataFrames.
columns_to_compare = [col for col in df_backup.columns if col in control_df.columns]

for col in columns_to_compare:
    s1 = control_df[col]
    s2 = df_backup[col]

    # Comparar valores, tratando np.nan como iguales si ambos lo son
    # Esto gestiona tanto valores iguales no nulos (s1 == s2) como coincidencias NaN-NaN (s1.isna() & s2.isna())
    matches = ((s1 == s2) | (s1.isna() & s2.isna()))
    matching_counts[col] = matches.sum()

# Crear un DataFrame para visualizar mejor los resultados
comparison_df = pd.DataFrame(matching_counts.items(), columns=['Column', 'Matching_Count'])
comparison_df['Total_Rows'] = len(df_backup)
comparison_df['Percentage_Match'] = (comparison_df['Matching_Count'] / comparison_df['Total_Rows']) * 100

display(comparison_df)

# Opcionalmente, mostrar columnas que no tienen un 100% de coincidencia para facilitar la inspección
non_full_match_cols = comparison_df[comparison_df['Percentage_Match'] < 100]
if not non_full_match_cols.empty:
    print("\nColumnas con menos del 100% de coincidencia (debido a diferencias reales de valores, no solo manejo de NaN):")
    display(non_full_match_cols)
else:
    print("\nTodas las columnas comparadas tienen una coincidencia del 100% (incluyendo la igualdad NaN-NaN).")

,Column,Matching_Count,Total_Rows,Percentage_Match
0,SK_ID_CURR,1716428,1716428,100.0
1,SK_ID_BUREAU,1716428,1716428,100.0
2,CREDIT_ACTIVE,1716428,1716428,100.0
3,CREDIT_CURRENCY,1716428,1716428,100.0
4,DAYS_CREDIT,1716428,1716428,100.0
5,CREDIT_DAY_OVERDUE,1716428,1716428,100.0
6,DAYS_CREDIT_ENDDATE,1716428,1716428,100.0
7,DAYS_ENDDATE_FACT,1716428,1716428,100.0
8,AMT_CREDIT_MAX_OVERDUE,1716428,1716428,100.0
9,CNT_CREDIT_PROLONG,1716428,1716428,100.0



Todas las columnas comparadas tienen una coincidencia del 100% (incluyendo la igualdad NaN-NaN).


### Exportación de DataFrames a CSV

Este paso guarda el DataFrame principal y cada una de las tablas de dimensiones creadas en archivos CSV individuales. Esto permite un almacenamiento y uso posterior más sencillo de los datos normalizados y las tablas de referencia.

In [62]:
# Creacion de directorio output.
ref_name = csv_name.replace('.csv', '')
extract_dir = output_dir + ref_name + '/'

# Evaluacion de existencia del directorio.
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir)

# Guardar el dataFrame principal 'df' en un archivo CSV.
df.to_csv(extract_dir + csv_name, index=False)
print(f'DataFrame principal guardado como {csv_name}')

# Guardar cada tabla de dimensiones en un archivo CSV separado
for name, dim_df in dimension_tables.items():
    file_name = f'{name}.csv'
    dim_df.to_csv(extract_dir + file_name, index=False)
    print(f'Tabla de dimensiones "{name}" guardada como {file_name}')

DataFrame principal guardado como bureau.csv
Tabla de dimensiones "dim_credit_active" guardada como dim_credit_active.csv
Tabla de dimensiones "dim_credit_currency" guardada como dim_credit_currency.csv
Tabla de dimensiones "dim_credit_type" guardada como dim_credit_type.csv


# credit_card_balance

### Analisis de datos preliminar





In [63]:
# Lectura del dataset.
csv_name = 'credit_card_balance.csv'
df = pd.read_csv(data_path + csv_name)

# Head del dataset.
df.info()

# Head del dataset.
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3840312 entries, 0 to 3840311
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   SK_ID_PREV                  int64  
 1   SK_ID_CURR                  int64  
 2   MONTHS_BALANCE              int64  
 3   AMT_BALANCE                 float64
 4   AMT_CREDIT_LIMIT_ACTUAL     int64  
 5   AMT_DRAWINGS_ATM_CURRENT    float64
 6   AMT_DRAWINGS_CURRENT        float64
 7   AMT_DRAWINGS_OTHER_CURRENT  float64
 8   AMT_DRAWINGS_POS_CURRENT    float64
 9   AMT_INST_MIN_REGULARITY     float64
 10  AMT_PAYMENT_CURRENT         float64
 11  AMT_PAYMENT_TOTAL_CURRENT   float64
 12  AMT_RECEIVABLE_PRINCIPAL    float64
 13  AMT_RECIVABLE               float64
 14  AMT_TOTAL_RECEIVABLE        float64
 15  CNT_DRAWINGS_ATM_CURRENT    float64
 16  CNT_DRAWINGS_CURRENT        int64  
 17  CNT_DRAWINGS_OTHER_CURRENT  float64
 18  CNT_DRAWINGS_POS_CURRENT    float64
 19  CNT_INSTALMENT_MATURE

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


In [64]:
# Obtención de valores únicos para atributos categóricos y numéricos no float.
for column in df.columns:
    unique_values = df[column].unique()

    # Indicador para verificar si la columna contiene algún valor float no entero (excluyendo NaN)
    contains_non_integer_float = False

    # Iterar sobre los valores únicos para encontrar floats no enteros
    for val in unique_values:
        # Verificar si el valor es un float y no es NaN
        if isinstance(val, float) and not np.isnan(val):
            # Verificar si el float tiene una parte decimal distinta de cero
            if val != int(val):
                contains_non_integer_float = True
                break  # Se encontró un float no entero, no es necesario seguir verificando

    # Si no se encontró ningún float no entero, imprimir los valores únicos de la columna
    if not contains_non_integer_float:
        print(f"Columna '{column}':{unique_values}")

Columna 'SK_ID_PREV':[2562384 2582071 1740877 ... 1897864 2481982 2191610]
Columna 'SK_ID_CURR':[378907 363914 371185 ... 174455 315041 382749]
Columna 'MONTHS_BALANCE':[ -6  -1  -7  -4  -5  -3  -2 -19 -13 -18 -15 -12 -22 -16 -21 -14 -20 -17
 -11 -52 -54 -58 -53 -51 -50 -59 -56 -57 -60 -55 -26 -25 -24 -23 -67 -64
 -62 -63 -61 -65 -66 -10  -9 -88 -95 -86 -89 -91 -93 -87 -90 -92 -94 -85
 -72 -73 -70 -69 -68 -71 -74  -8 -27 -80 -79 -84 -83 -81 -82 -96 -28 -36
 -31 -32 -40 -41 -35 -34 -39 -33 -37 -38 -47 -48 -45 -44 -43 -49 -42 -46
 -76 -78 -77 -75 -30 -29]
Columna 'AMT_CREDIT_LIMIT_ACTUAL':[ 135000   45000  450000  225000  270000  585000  292500  180000   67500
       0  157500  360000  540000  558000  202500  247500  112500   90000
  720000  315000  900000   22500  675000  855000  382500  495000  337500
  765000  405000  108000  630000  810000  427500  261000  229500   27000
   81000  216000  103500   58500  468000  342000   94500  274500  234000
  130500  400500    4500   36000  306000 

In [65]:
# Obtención de valores únicos para atributos categóricos.
for column in df.columns:
    # Evaluación si el tipo es string.
    if df[column].dtype == 'object':
        unique_values = df[column].unique()
        print(f"Columna '{column}': {unique_values}")

Columna 'NAME_CONTRACT_STATUS': ['Active' 'Completed' 'Demand' 'Signed' 'Sent proposal' 'Refused'
 'Approved']


### Mejoras de diseño

Dado que un sólo atributo es optimizable, se mantienen los datos en la tabla en el mismo estado en que se encuentran.

# installments_payments

### Analisis de datos preliminar





In [66]:
# Lectura del dataset.
csv_name = 'installments_payments.csv'
df = pd.read_csv(data_path + csv_name)

# Head del dataset.
df.info()

# Head del dataset.
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 8 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int64  
 1   SK_ID_CURR              int64  
 2   NUM_INSTALMENT_VERSION  float64
 3   NUM_INSTALMENT_NUMBER   int64  
 4   DAYS_INSTALMENT         float64
 5   DAYS_ENTRY_PAYMENT      float64
 6   AMT_INSTALMENT          float64
 7   AMT_PAYMENT             float64
dtypes: float64(5), int64(3)
memory usage: 830.4 MB


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [67]:
# Obtención de valores únicos para atributos categóricos y numéricos no float.
for column in df.columns:
    unique_values = df[column].unique()

    # Indicador para verificar si la columna contiene algún valor float no entero (excluyendo NaN)
    contains_non_integer_float = False

    # Iterar sobre los valores únicos para encontrar floats no enteros
    for val in unique_values:
        # Verificar si el valor es un float y no es NaN
        if isinstance(val, float) and not np.isnan(val):
            # Verificar si el float tiene una parte decimal distinta de cero
            if val != int(val):
                contains_non_integer_float = True
                break  # Se encontró un float no entero, no es necesario seguir verificando

    # Si no se encontró ningún float no entero, imprimir los valores únicos de la columna
    if not contains_non_integer_float:
        print(f"Columna '{column}':{unique_values}")

Columna 'SK_ID_PREV':[1054186 1330831 2085231 ... 1984920 1946334 1276288]
Columna 'SK_ID_CURR':[161674 151639 193053 ... 413433 434445 405063]
Columna 'NUM_INSTALMENT_VERSION':[  1.   0.   2.   4.   3.   5.   7.   8.   6.  13.   9.  21.  22.  12.
  17.  18.  11.  14.  34.  33.  19.  16.  15.  10.  26.  27.  20.  25.
  23.  24.  31.  32.  28.  35.  29.  30.  43.  39.  40.  36.  41.  42.
  37.  38.  68.  44.  45.  46. 178.  52.  51.  53.  54.  49.  50.  58.
  57.  55.  56.  48.  47.  72.  59.  73.  61.]
Columna 'NUM_INSTALMENT_NUMBER':[  6  34   1   3   2  12  11   4  14   8  20  56   7  23  38 116   5  46
 106   9 105  88  30  16  65  19  15  57  10  41  84  49  59 107  17  94
  64  28  24 140  43  99  69  13  22  85  29  83  73 139  39  18  26  25
  21  44  27  31  37  63  45  78  53  92 112  36  93 114  58  32  67  47
  68  40  81  61 117  60  75 149  35  48  82  79  71  51  72  80 156  42
  50 148  33  55 103  54 180  86 113  87  62 102 115 142 164 162 101  91
  96 100 104 121  66 1

In [68]:
# Obtención de valores únicos para atributos categóricos.
for column in df.columns:
    # Evaluación si el tipo es string.
    if df[column].dtype == 'object':
        unique_values = df[column].unique()
        print(f"Columna '{column}': {unique_values}")

### Mejoras de diseño

Dado que un sólo atributo es optimizable, se mantienen los datos en la tabla en el mismo estado en que se encuentran.

### Exportación de DataFrames a CSV

Este paso guarda el DataFrame principal y cada una de las tablas de dimensiones creadas en archivos CSV individuales. Esto permite un almacenamiento y uso posterior más sencillo de los datos normalizados y las tablas de referencia.

In [ ]:
# Creacion de directorio output.
ref_name = csv_name.replace('.csv', '')
extract_dir = output_dir + ref_name + '/'

# Evaluacion de existencia del directorio.
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir)

# Guardar el dataFrame principal 'df' en un archivo CSV.
df.to_csv(extract_dir + csv_name, index=False)
print(f'DataFrame principal guardado como {csv_name}')

# Guardar cada tabla de dimensiones en un archivo CSV separado
for name, dim_df in dimension_tables.items():
    file_name = f'{name}.csv'
    dim_df.to_csv(extract_dir + file_name, index=False)
    print(f'Tabla de dimensiones "{name}" guardada como {file_name}')

# POS_CASH_balance

### Analisis de datos preliminar





In [69]:
# Lectura del dataset.
csv_name = 'POS_CASH_balance.csv'
df = pd.read_csv(data_path + csv_name)

# Head del dataset.
df.info()

# Head del dataset.
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10001358 entries, 0 to 10001357
Data columns (total 8 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   SK_ID_PREV             int64  
 1   SK_ID_CURR             int64  
 2   MONTHS_BALANCE         int64  
 3   CNT_INSTALMENT         float64
 4   CNT_INSTALMENT_FUTURE  float64
 5   NAME_CONTRACT_STATUS   object 
 6   SK_DPD                 int64  
 7   SK_DPD_DEF             int64  
dtypes: float64(2), int64(5), object(1)
memory usage: 610.4+ MB


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [70]:
# Obtención de valores únicos para atributos categóricos y numéricos no float.
for column in df.columns:
    unique_values = df[column].unique()

    # Indicador para verificar si la columna contiene algún valor float no entero (excluyendo NaN)
    contains_non_integer_float = False

    # Iterar sobre los valores únicos para encontrar floats no enteros
    for val in unique_values:
        # Verificar si el valor es un float y no es NaN
        if isinstance(val, float) and not np.isnan(val):
            # Verificar si el float tiene una parte decimal distinta de cero
            if val != int(val):
                contains_non_integer_float = True
                break  # Se encontró un float no entero, no es necesario seguir verificando

    # Si no se encontró ningún float no entero, imprimir los valores únicos de la columna
    if not contains_non_integer_float:
        print(f"Columna '{column}':{unique_values}")

Columna 'SK_ID_PREV':[1803195 1715348 1784872 ... 2452932 1980876 2308284]
Columna 'SK_ID_CURR':[182943 367990 397406 ... 394310 240059 144536]
Columna 'MONTHS_BALANCE':[-31 -33 -32 -35 -38 -39 -34 -41 -37 -40 -43 -36 -42 -47 -44 -46 -45 -27
 -25 -24 -30 -28 -26 -29 -48 -15 -19 -22 -14 -18 -16 -21 -17 -20 -23  -1
  -6  -2  -3  -5  -4 -13 -11 -10  -9  -7  -8 -54 -49 -52 -53 -50 -51 -55
 -12 -94 -95 -92 -91 -96 -93 -57 -64 -56 -59 -60 -66 -65 -58 -63 -62 -61
 -87 -86 -88 -89 -90 -72 -74 -73 -82 -78 -81 -75 -77 -80 -76 -79 -69 -70
 -71 -68 -67 -83 -85 -84]
Columna 'CNT_INSTALMENT':[48. 36. 12. 24. 60. 18.  4. 42. 25. 14. 16. 13.  8. 10. 15. 11. 30. 54.
  6.  9.  5. 17.  3.  2. 20. 32. nan  1.  7. 47. 49. 28. 43. 23. 21. 19.
 39. 37. 35. 27. 22. 41. 31. 61. 26. 34. 29. 44. 55. 33. 66. 46. 59. 38.
 72. 40. 57. 45. 92. 56. 84. 51. 52. 58. 62. 53. 64. 50. 77. 70. 81. 63.
 71. 68.]
Columna 'CNT_INSTALMENT_FUTURE':[45. 35.  9. 42. 12. 43. 36. 16. 24.  5. 15.  1. 28. 23. 56. 11.  7. 18.
 17. 46.

In [71]:
# Obtención de valores únicos para atributos categóricos.
for column in df.columns:
    # Evaluación si el tipo es string.
    if df[column].dtype == 'object':
        unique_values = df[column].unique()
        print(f"Columna '{column}': {unique_values}")

Columna 'NAME_CONTRACT_STATUS': ['Active' 'Completed' 'Signed' 'Approved' 'Returned to the store' 'Demand'
 'Canceled' 'XNA' 'Amortized debt']


### Mejoras de diseño

Dado que un sólo atributo es optimizable, se mantienen los datos en la tabla en el mismo estado en que se encuentran.

### Exportación de DataFrames a CSV

Este paso guarda el DataFrame principal y cada una de las tablas de dimensiones creadas en archivos CSV individuales. Esto permite un almacenamiento y uso posterior más sencillo de los datos normalizados y las tablas de referencia.

In [ ]:
# Creacion de directorio output.
ref_name = csv_name.replace('.csv', '')
extract_dir = output_dir + ref_name + '/'

# Evaluacion de existencia del directorio.
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir)

# Guardar el dataFrame principal 'df' en un archivo CSV.
df.to_csv(extract_dir + csv_name, index=False)
print(f'DataFrame principal guardado como {csv_name}')

# Guardar cada tabla de dimensiones en un archivo CSV separado
for name, dim_df in dimension_tables.items():
    file_name = f'{name}.csv'
    dim_df.to_csv(extract_dir + file_name, index=False)
    print(f'Tabla de dimensiones "{name}" guardada como {file_name}')

# previous_application

### Analisis de datos preliminar





In [72]:
# Lectura del dataset.
csv_name = 'previous_application.csv'
df = pd.read_csv(data_path + csv_name)

# Head del dataset.
df.info()

# Head del dataset.
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   SK_ID_PREV                   1670214 non-null  int64  
 1   SK_ID_CURR                   1670214 non-null  int64  
 2   NAME_CONTRACT_TYPE           1670214 non-null  object 
 3   AMT_ANNUITY                  1297979 non-null  float64
 4   AMT_APPLICATION              1670214 non-null  float64
 5   AMT_CREDIT                   1670213 non-null  float64
 6   AMT_DOWN_PAYMENT             774370 non-null   float64
 7   AMT_GOODS_PRICE              1284699 non-null  float64
 8   WEEKDAY_APPR_PROCESS_START   1670214 non-null  object 
 9   HOUR_APPR_PROCESS_START      1670214 non-null  int64  
 10  FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  object 
 11  NFLAG_LAST_APPL_IN_DAY       1670214 non-null  int64  
 12  RATE_DOWN_PAYMENT            774370 non-nu

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [73]:
# Obtención de valores únicos para atributos categóricos y numéricos no float.
for column in df.columns:
    unique_values = df[column].unique()

    # Indicador para verificar si la columna contiene algún valor float no entero (excluyendo NaN)
    contains_non_integer_float = False

    # Iterar sobre los valores únicos para encontrar floats no enteros
    for val in unique_values:
        # Verificar si el valor es un float y no es NaN
        if isinstance(val, float) and not np.isnan(val):
            # Verificar si el float tiene una parte decimal distinta de cero
            if val != int(val):
                contains_non_integer_float = True
                break  # Se encontró un float no entero, no es necesario seguir verificando

    # Si no se encontró ningún float no entero, imprimir los valores únicos de la columna
    if not contains_non_integer_float:
        print(f"Columna '{column}':{unique_values}")

Columna 'SK_ID_PREV':[2030495 2802425 2523466 ... 2659632 2785582 2418762]
Columna 'SK_ID_CURR':[271877 108129 122040 ... 253940 353284 191629]
Columna 'NAME_CONTRACT_TYPE':['Consumer loans' 'Cash loans' 'Revolving loans' 'XNA']
Columna 'WEEKDAY_APPR_PROCESS_START':['SATURDAY' 'THURSDAY' 'TUESDAY' 'MONDAY' 'FRIDAY' 'SUNDAY' 'WEDNESDAY']
Columna 'HOUR_APPR_PROCESS_START':[15 11  7  9  8 10 12 13 14 16  6  4  5 19 17 18 20 22 21  3  1  2 23  0]
Columna 'FLAG_LAST_APPL_PER_CONTRACT':['Y' 'N']
Columna 'NFLAG_LAST_APPL_IN_DAY':[1 0]
Columna 'NAME_CASH_LOAN_PURPOSE':['XAP' 'XNA' 'Repairs' 'Everyday expenses' 'Car repairs'
 'Building a house or an annex' 'Other' 'Journey'
 'Purchase of electronic equipment' 'Medicine' 'Payments on other loans'
 'Urgent needs' 'Buying a used car' 'Buying a new car'
 'Buying a holiday home / land' 'Education' 'Buying a home' 'Furniture'
 'Buying a garage' 'Business development' 'Wedding / gift / holiday'
 'Hobby' 'Gasification / water supply' 'Refusal to name t

In [74]:
# Obtención de valores únicos para atributos categóricos.
for column in df.columns:
    # Evaluación si el tipo es string.
    if df[column].dtype == 'object':
        unique_values = df[column].unique()
        print(f"Columna '{column}': {unique_values}")

Columna 'NAME_CONTRACT_TYPE': ['Consumer loans' 'Cash loans' 'Revolving loans' 'XNA']
Columna 'WEEKDAY_APPR_PROCESS_START': ['SATURDAY' 'THURSDAY' 'TUESDAY' 'MONDAY' 'FRIDAY' 'SUNDAY' 'WEDNESDAY']
Columna 'FLAG_LAST_APPL_PER_CONTRACT': ['Y' 'N']
Columna 'NAME_CASH_LOAN_PURPOSE': ['XAP' 'XNA' 'Repairs' 'Everyday expenses' 'Car repairs'
 'Building a house or an annex' 'Other' 'Journey'
 'Purchase of electronic equipment' 'Medicine' 'Payments on other loans'
 'Urgent needs' 'Buying a used car' 'Buying a new car'
 'Buying a holiday home / land' 'Education' 'Buying a home' 'Furniture'
 'Buying a garage' 'Business development' 'Wedding / gift / holiday'
 'Hobby' 'Gasification / water supply' 'Refusal to name the goal'
 'Money for a third person']
Columna 'NAME_CONTRACT_STATUS': ['Approved' 'Refused' 'Canceled' 'Unused offer']
Columna 'NAME_PAYMENT_TYPE': ['Cash through the bank' 'XNA' 'Non-cash from your account'
 'Cashless from the account of the employer']
Columna 'CODE_REJECT_REASON': ['X

### Mejoras de diseño

In [75]:
# Creación de copia para control posterior.
df_backup = df.copy()

In [76]:
# Obtención de atributos categoricos.
dimension_tables = {}
categorical_columns = [col for col in df.columns if df[col].dtype == 'object']

In [77]:
# Creación de tablas de dimensiones.
for col in categorical_columns:
    # Extraer valores únicos no NaN para la tabla de dimensión
    unique_values = df[col].dropna().unique()

    if len(unique_values) > 0:
        # Crear una tabla de dimensión para la columna categórica actual
        dim_df = pd.DataFrame({
            f'{col}_ID': range(1, len(unique_values) + 1),
            col: unique_values
        })
        dimension_tables[f'dim_{col.lower()}'] = dim_df

        # Crear un mapeo desde el valor original al nuevo ID, manejando NaN
        mapping = dict(zip(dim_df[col], dim_df[f'{col}_ID']))

        # Reemplazar los valores categóricos en el DataFrame original con sus IDs
        # Usar .map() que maneja valores NaN por defecto, preservándolos si no están en el mapeo
        df[col] = df[col].map(mapping)

        # Renombrar la columna para reflejar que ahora contiene IDs
        df.rename(columns={col: f'{col}_ID'}, inplace=True)
    else:
        print(f"La columna '{col}' no tiene valores únicos que no estén vacios, por lo tanto se omite la creación de la tabla de dimensión.")

In [78]:
# Control de atributos.
print(f'Recuento de columnas categoricas: {len(categorical_columns)}')
print(f'Recuento de tablas de dimensiones: {len(dimension_tables)}')

Recuento de columnas categoricas: 16
Recuento de tablas de dimensiones: 16


In [79]:
# Tablas de dimensiones.
print(f'Tablas de dimensiones: {[*dimension_tables.keys()]}')

Tablas de dimensiones: ['dim_name_contract_type', 'dim_weekday_appr_process_start', 'dim_flag_last_appl_per_contract', 'dim_name_cash_loan_purpose', 'dim_name_contract_status', 'dim_name_payment_type', 'dim_code_reject_reason', 'dim_name_type_suite', 'dim_name_client_type', 'dim_name_goods_category', 'dim_name_portfolio', 'dim_name_product_type', 'dim_channel_type', 'dim_name_seller_industry', 'dim_name_yield_group', 'dim_product_combination']


In [80]:
# Print de atributos del dataset backup.
print(f'Columnas del dataset de referencia: {[*df_backup.columns]}')

Columnas del dataset de referencia: ['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY', 'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY', 'CNT_PAYMENT', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION', 'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION', 'NFLAG_INSURED_ON_APPROVAL']


In [81]:
# Query para la verificación del join de tablas principal y de dimensiones.
query = f"""
SELECT {', '.join(df_backup.columns)}
FROM df
"""

dim_tables_names = [*dimension_tables.keys()]

for dim_table_name in dim_tables_names:
  dim_table_id = dimension_tables[dim_table_name].columns[0]

  query += f"""
  LEFT JOIN {dim_table_name} ON df.{dim_table_id} = {dim_table_name}.{dim_table_id}
  """

In [82]:
# Impresion de la query.
print(query)


SELECT SK_ID_PREV, SK_ID_CURR, NAME_CONTRACT_TYPE, AMT_ANNUITY, AMT_APPLICATION, AMT_CREDIT, AMT_DOWN_PAYMENT, AMT_GOODS_PRICE, WEEKDAY_APPR_PROCESS_START, HOUR_APPR_PROCESS_START, FLAG_LAST_APPL_PER_CONTRACT, NFLAG_LAST_APPL_IN_DAY, RATE_DOWN_PAYMENT, RATE_INTEREST_PRIMARY, RATE_INTEREST_PRIVILEGED, NAME_CASH_LOAN_PURPOSE, NAME_CONTRACT_STATUS, DAYS_DECISION, NAME_PAYMENT_TYPE, CODE_REJECT_REASON, NAME_TYPE_SUITE, NAME_CLIENT_TYPE, NAME_GOODS_CATEGORY, NAME_PORTFOLIO, NAME_PRODUCT_TYPE, CHANNEL_TYPE, SELLERPLACE_AREA, NAME_SELLER_INDUSTRY, CNT_PAYMENT, NAME_YIELD_GROUP, PRODUCT_COMBINATION, DAYS_FIRST_DRAWING, DAYS_FIRST_DUE, DAYS_LAST_DUE_1ST_VERSION, DAYS_LAST_DUE, DAYS_TERMINATION, NFLAG_INSURED_ON_APPROVAL
FROM df

  LEFT JOIN dim_name_contract_type ON df.NAME_CONTRACT_TYPE_ID = dim_name_contract_type.NAME_CONTRACT_TYPE_ID
  
  LEFT JOIN dim_weekday_appr_process_start ON df.WEEKDAY_APPR_PROCESS_START_ID = dim_weekday_appr_process_start.WEEKDAY_APPR_PROCESS_START_ID
  
  LEFT JOIN

In [83]:
# Control de join de tablas descompuestas.

# Construcción de diccionario con dataframe principal y dataframes de dimensiones.
env = {**dimension_tables, 'df': df}

# Ejecución de la query.
control_df = sqldf(query, env) # sqldf(query, locals())
control_df.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [84]:
# Shape de control_df.
control_df.shape

(1670214, 37)

In [85]:
# Reemplazo de NoneType por np.nans.
for col in control_df.columns:
    # Verificacion si 'None' (NoneType) está presente en la columna.
    if control_df[col].apply(lambda x: x is None).any():
        control_df[col] = control_df[col].replace({None: np.nan})

print("Valores None fueron reemplazados con np.nan para todas las columnas de control_df.")

Valores None fueron reemplazados con np.nan para todas las columnas de control_df.


In [86]:
# Cotejo final de control_df con df_backup.

matching_counts = {}

# Asegurar que solo comparamos las columnas que existen en ambos DataFrames.
columns_to_compare = [col for col in df_backup.columns if col in control_df.columns]

for col in columns_to_compare:
    s1 = control_df[col]
    s2 = df_backup[col]

    # Comparar valores, tratando np.nan como iguales si ambos lo son
    # Esto gestiona tanto valores iguales no nulos (s1 == s2) como coincidencias NaN-NaN (s1.isna() & s2.isna())
    matches = ((s1 == s2) | (s1.isna() & s2.isna()))
    matching_counts[col] = matches.sum()

# Crear un DataFrame para visualizar mejor los resultados
comparison_df = pd.DataFrame(matching_counts.items(), columns=['Column', 'Matching_Count'])
comparison_df['Total_Rows'] = len(df_backup)
comparison_df['Percentage_Match'] = (comparison_df['Matching_Count'] / comparison_df['Total_Rows']) * 100

display(comparison_df)

# Opcionalmente, mostrar columnas que no tienen un 100% de coincidencia para facilitar la inspección
non_full_match_cols = comparison_df[comparison_df['Percentage_Match'] < 100]
if not non_full_match_cols.empty:
    print("\nColumnas con menos del 100% de coincidencia (debido a diferencias reales de valores, no solo manejo de NaN):")
    display(non_full_match_cols)
else:
    print("\nTodas las columnas comparadas tienen una coincidencia del 100% (incluyendo la igualdad NaN-NaN).")

,Column,Matching_Count,Total_Rows,Percentage_Match
0,SK_ID_PREV,1670214,1670214,100.0
1,SK_ID_CURR,1670214,1670214,100.0
2,NAME_CONTRACT_TYPE,1670214,1670214,100.0
3,AMT_ANNUITY,1670214,1670214,100.0
4,AMT_APPLICATION,1670214,1670214,100.0
5,AMT_CREDIT,1670214,1670214,100.0
6,AMT_DOWN_PAYMENT,1670214,1670214,100.0
7,AMT_GOODS_PRICE,1670214,1670214,100.0
8,WEEKDAY_APPR_PROCESS_START,1670214,1670214,100.0
9,HOUR_APPR_PROCESS_START,1670214,1670214,100.0



Todas las columnas comparadas tienen una coincidencia del 100% (incluyendo la igualdad NaN-NaN).


### Exportación de DataFrames a CSV

Este paso guarda el DataFrame principal y cada una de las tablas de dimensiones creadas en archivos CSV individuales. Esto permite un almacenamiento y uso posterior más sencillo de los datos normalizados y las tablas de referencia.

In [87]:
# Creacion de directorio output.
ref_name = csv_name.replace('.csv', '')
extract_dir = output_dir + ref_name + '/'

# Evaluacion de existencia del directorio.
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir)

# Guardar el dataFrame principal 'df' en un archivo CSV.
df.to_csv(extract_dir + csv_name, index=False)
print(f'DataFrame principal guardado como {csv_name}')

# Guardar cada tabla de dimensiones en un archivo CSV separado
for name, dim_df in dimension_tables.items():
    file_name = f'{name}.csv'
    dim_df.to_csv(extract_dir + file_name, index=False)
    print(f'Tabla de dimensiones "{name}" guardada como {file_name}')

DataFrame principal guardado como previous_application.csv
Tabla de dimensiones "dim_name_contract_type" guardada como dim_name_contract_type.csv
Tabla de dimensiones "dim_weekday_appr_process_start" guardada como dim_weekday_appr_process_start.csv
Tabla de dimensiones "dim_flag_last_appl_per_contract" guardada como dim_flag_last_appl_per_contract.csv
Tabla de dimensiones "dim_name_cash_loan_purpose" guardada como dim_name_cash_loan_purpose.csv
Tabla de dimensiones "dim_name_contract_status" guardada como dim_name_contract_status.csv
Tabla de dimensiones "dim_name_payment_type" guardada como dim_name_payment_type.csv
Tabla de dimensiones "dim_code_reject_reason" guardada como dim_code_reject_reason.csv
Tabla de dimensiones "dim_name_type_suite" guardada como dim_name_type_suite.csv
Tabla de dimensiones "dim_name_client_type" guardada como dim_name_client_type.csv
Tabla de dimensiones "dim_name_goods_category" guardada como dim_name_goods_category.csv
Tabla de dimensiones "dim_name_port